In [1]:
# Imports 
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.pyplot import cm
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual, Layout, HBox, VBox
from IPython.display import clear_output
import math
from scipy.optimize import fsolve
import seaborn as sns
import openpyxl
from types import NoneType
import bokeh
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
import matplotlib.lines as mlines

from bokeh.util.warnings import BokehDeprecationWarning, BokehUserWarning

# Ignore specific Bokeh validation and deprecation warnings
warnings.simplefilter(action='ignore', category=BokehUserWarning)
warnings.simplefilter(action='ignore', category=BokehDeprecationWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
output_notebook()

cases_dict = {'No Liquid Immiscibility':0, 'Equilibrium':1, 'S-Rich Crystalliation':2, 'P-Rich Crystallization':3}


Loading BokehJS ...

In [2]:
import model.functions
from model.run_crystallization import run_crystallization

In [3]:
# set figure styling
colors = sns.color_palette("colorblind").as_hex()

plt.rcParams['font.size'] = 18
plt.rc('axes', titlesize=24)  # Font size for axis titles
plt.rc('axes', labelsize=20)  # Font size for axis labels
plt.rc('xtick', labelsize=18)  # Font size for x-axis tick labels
plt.rc('ytick', labelsize=18)  # Font size for y-axis tick labels
plt.rc('legend', fontsize=18)

linesize = 4

markerSize = 5
opac = 0.7
numCols = 4
log_scale = True
fig_width = 750
fig_height = 600

In [4]:
from bokeh.plotting import figure
from bokeh.models import Legend, LegendItem, HoverTool
from model.functions import make_mixing_lines, get_meteorite_coords
global fig, custom_points_plot


def add_hover_tool(fig_info, crystallization, L1S, L1P, L1Ni, L2S, L2P, L2Ni, bulk_L_S, bulk_L_P, bulk_L_Ni):
    fig_info.data_source.data['per_cry'] = crystallization*100
    fig_info.data_source.data['bulk_L_S'] = bulk_L_S
    fig_info.data_source.data['bulk_L_P'] = bulk_L_P
    fig_info.data_source.data['bulk_L_Ni'] = bulk_L_Ni
    fig_info.data_source.data['L1S'] = L1S
    fig_info.data_source.data['L1P'] = L1P
    fig_info.data_source.data['L1Ni'] = L1Ni
    fig_info.data_source.data['L2S'] = L2S
    fig_info.data_source.data['L2P'] = L2P
    fig_info.data_source.data['L2Ni'] = L2Ni
    line_hover = HoverTool(renderers=[fig_info], tooltips=[("(x,y)", "($x, $y)"), ("Percent Crystallization", "@per_cry"), 
                                                           ("S-Rich L (S, P, Ni) (wt%)", "(@L1S, @L1P, @L1Ni)"), 
                                                           ("P-Rich L (S, P, Ni) (wt%)", "(@L2S, @L2P, @L2Ni)"), 
                                                           ("Bulk L (S, P, Ni) (wt%)", "(@bulk_L_S, @bulk_L_P, @bulk_L_Ni)")])        

    return line_hover

def add_mixing_hover(fig_info):
    percent_solid = [100-i for i in range(0, 110, 10)]
    percent_trapped = [i for i in range(0, 110, 10)]
    fig_info.data_source.data['PerSolid'] = percent_solid
    fig_info.data_source.data['PerTrapped'] = percent_trapped
    mix_hover = HoverTool(renderers=[fig_info], tooltips=[("(x,y)", "($x, $y)"), ("(% Solid, % Trapped Solid)", "(@PerSolid, @PerTrapped)")]) 
    
    return mix_hover

    
def generate_bokeh_plot(state_data, data1, data2, opt1, opt2, unitx, unity, mGroup, mixing_opt, liq_imm_case):
    two_liq = True

    opt1 = opt1.value
    opt2 = opt2.value
    unitx = unitx.value
    unity = unity.value
    mGroup = mGroup.value
    mixing_opt = mixing_opt.value
    
    ele1_sm = data1[f'SM {opt1}']
    ele1_trapped = data1[f'Trapped SM {opt1}']
    ele1_sm_2liq_S1 = data1[f'S1 {opt1}']
    ele1_sm_2liq = data1[f'S2 {opt1}']
    ele1_tm1_2liq = data1[f'Trapped 1 {opt1}']
    ele1_tm2_2liq = data1[f'Trapped 2 {opt1}']
    ele1_bulk_trapped = data1[f'Bulk Trapped {opt1}']

    ele2_sm = data2[f'SM {opt2}']
    ele2_trapped = data2[f'Trapped SM {opt2}']
    ele2_sm_2liq_S1 = data2[f'S1 {opt2}']
    ele2_sm_2liq = data2[f'S2 {opt2}']
    ele2_tm1_2liq = data2[f'Trapped 1 {opt2}']
    ele2_tm2_2liq = data2[f'Trapped 2 {opt2}']
    ele2_bulk_trapped = data2[f'Bulk Trapped {opt2}']

    # convert to np arrays for easier indexing
    ele1_tm1_2liq_array = np.array(ele1_tm1_2liq)
    ele1_tm2_2liq_array = np.array(ele1_tm2_2liq)
    ele2_tm1_2liq_array = np.array(ele2_tm1_2liq)
    ele2_tm2_2liq_array = np.array(ele2_tm2_2liq)
    ele1_bulk_trapped_array = np.array(ele1_bulk_trapped)
    ele2_bulk_trapped_array = np.array(ele2_bulk_trapped)
    ele1_sm_2liq_array = np.array(ele1_sm_2liq)
    ele2_sm_2liq_array = np.array(ele2_sm_2liq)

    LM_S = state_data['LM_S']
    LM_P = state_data['LM_P']
    LM_Ni = state_data['LM_Ni']
    
    S_start = LM_S[0]
    P_start = LM_P[0]
    Ni_start = LM_Ni[0]
    E1_start = data1[f'LM {opt1}'][0]
    E2_start = data2[f'LM {opt2}'][0]
    crystallization = np.array(state_data['Percent Crystallization'])

    bulk_L_S = state_data['Bulk L_S']
    bulk_L_P = state_data['Bulk L_P']
    bulk_L_Ni = state_data['Bulk L_Ni']
    L1S = state_data['LM1_S']
    L1P = state_data['LM1_P']
    L1Ni = state_data['LM1_Ni']
    L2S = state_data['LM2_S']
    L2P = state_data['LM2_P']
    L2Ni = state_data['LM2_Ni']

    # # MIXING LINE INDICES
    validIdxs = np.where(np.isnan(ele1_sm))

    # one liquid mixing line indices
    # mixing line 1: very top of curve - 0% crystallized
    valid1_1liq = 1
    # mixing line 2: very bottom of one liquid curve (validIdxs[0][1] - 1 = the index right before 2 liquid begins)
    try:
        valid2_1liq = validIdxs[0][1] - 1
    except:
        two_liq = False
        valid2_1liq = len(ele1_sm)-1

    if mixing_opt == 'Default':
        mixing_dict = make_mixing_lines(data1, data2, opt1, opt2)
    elif mixing_opt == 'Fixed Number':
        mixing_dict = make_mixing_lines(data1, data2, opt1, opt2, num_lines=num_lines_box.value)
    else:
        percents_string = per_cry_text.value
        split_percents = percents_string.split(",")
        float_fracs = [float(percent)/100 for percent in split_percents]
        mixing_dict = make_mixing_lines(data1, data2, opt1, opt2, per_crystallized=float_fracs)
        return_percents = ''
        max_percent = crystallization[-1]
        for frac in float_fracs:
            if frac < max_percent:
                return_percents = f'{return_percents}{frac*100:.2f}, '
            else:
                return_percents = f'{return_percents}{max_percent*100:.f}, '
            
        per_cry_text.value = return_percents[0:-2]


    fig = figure(width=fig_width, height=fig_height, x_axis_label=f'{opt1} ({unitx})', y_axis_label=f'{opt2} ({unity})',
                 x_axis_type="log", y_axis_type="log", tools='pan,box_select,box_zoom,lasso_select,reset,save,zoom_in,zoom_out,wheel_zoom')
                 # tooltips=TOOLTIPS) # figsize=(15,10)


    custom_points_plot = None

    if valid2_1liq > 1:
        # one liquid SM
        one_liq_plot = fig.line(ele1_sm, ele2_sm, line_color=colors[4], line_width=linesize,legend_label='Solid Metal (One Liquid Field)')
        oneliq_hover = add_hover_tool(one_liq_plot, crystallization, L1S, L1P, L1Ni, L2S, L2P, L2Ni, bulk_L_S, bulk_L_P, bulk_L_Ni)
        fig.add_tools(oneliq_hover)
        # one liquid trapped
        one_liq_trap_plot = fig.line(ele1_trapped, ele2_trapped, line_color=colors[4],  line_width=linesize, line_dash='dashed', legend_label='Trapped Metal')
        oneliqtrap_hover = add_hover_tool(one_liq_trap_plot, crystallization, L1S, L1P, L1Ni, L2S, L2P, L2Ni, bulk_L_S, bulk_L_P, bulk_L_Ni)
        fig.add_tools(oneliqtrap_hover)
        # one liquid mixing lines
        one_liq_lines = mixing_dict['One Liquid']
        for per_cry, opts in one_liq_lines.items():
            ele1_mix = one_liq_lines[per_cry][f'{opt1}']
            ele2_mix = one_liq_lines[per_cry][f'{opt2}']
            oneliq_mix = fig.line(ele1_mix, ele2_mix, line_color=colors[4], line_width=linesize, line_dash="dotted", legend_label="One Liquid Mixing Lines")
            onqliq_hover = add_mixing_hover(oneliq_mix)
            fig.add_tools(onqliq_hover)

    if two_liq:
        # two liquid SM
        two_liq_plot = fig.line(ele1_sm_2liq_array, ele2_sm_2liq_array, line_color=colors[2], line_width=linesize, legend_label='Solid Metal (Two Liquid Field)')
        twoliq_hover = add_hover_tool(two_liq_plot, crystallization, L1S, L1P, L1Ni, L2S, L2P, L2Ni, bulk_L_S, bulk_L_P, bulk_L_Ni)          
        fig.add_tools(twoliq_hover)
        # two liquid bulk trapped
        two_liq_trap_plot = fig.line(ele1_bulk_trapped_array, ele2_bulk_trapped_array, line_color=colors[2], line_width=linesize, line_dash='dashdot', legend_label='Bulk Trapped')
        twoliqtrap_hover = add_hover_tool(two_liq_trap_plot, crystallization, L1S, L1P, L1Ni, L2S, L2P, L2Ni, bulk_L_S, bulk_L_P, bulk_L_Ni)    
        fig.add_tools(twoliqtrap_hover)
        # Bulk Mixing
        bulk_liq_lines = mixing_dict['Bulk Liquid']
        for per_cry, opts in bulk_liq_lines.items():
            ele1_mix = bulk_liq_lines[per_cry][f'{opt1}']
            ele2_mix = bulk_liq_lines[per_cry][f'{opt2}']
            bulk_mix = fig.line(ele1_mix, ele2_mix, line_color=colors[2], line_width=linesize, line_dash="dotted", legend_label="Bulk Mixing Lines")
            bulk_mix_hover = add_mixing_hover(bulk_mix)
            fig.add_tools(bulk_mix_hover)

        # S-Rich trapped
        s_trap_plot = fig.line(ele1_tm1_2liq_array, ele2_tm1_2liq_array, line_color=colors[1], line_width=linesize, line_dash='dashdot',
                 line_alpha=opac, legend_label='Solid from trapped S-rich Liquid')
        s_trap_hover = add_hover_tool(s_trap_plot, crystallization, L1S, L1P, L1Ni, L2S, L2P, L2Ni, bulk_L_S, bulk_L_P, bulk_L_Ni)          
        fig.add_tools(s_trap_hover)
        # S-Rich mixing
        s_liq_lines = mixing_dict['S-Rich Liquid']
        for per_cry, opts in s_liq_lines.items():
            ele1_mix = s_liq_lines[per_cry][f'{opt1}']
            ele2_mix = s_liq_lines[per_cry][f'{opt2}']
            s_mix = fig.line(ele1_mix, ele2_mix, line_color=colors[1], line_width=linesize, line_dash="dotted", line_alpha=opac, legend_label="S-Rich Mixing Lines")
            s_mix_hover = add_mixing_hover(s_mix)
            fig.add_tools(s_mix_hover)
            
        # P-Rich trapped
        p_trap_plot = fig.line(ele1_tm2_2liq_array, ele2_tm2_2liq_array, line_color=colors[0], line_width=linesize, line_dash='dashdot',
                 line_alpha=opac, legend_label='Solid from trapped P-rich Liquid')
        p_trap_hover = add_hover_tool(p_trap_plot, crystallization, L1S, L1P, L1Ni, L2S, L2P, L2Ni, bulk_L_S, bulk_L_P, bulk_L_Ni)                      
        fig.add_tools(p_trap_hover)
        # P-Rich mixing
        p_liq_lines = mixing_dict['P-Rich Liquid']
        for per_cry, opts in p_liq_lines.items():
            ele1_mix = p_liq_lines[per_cry][f'{opt1}']
            ele2_mix = p_liq_lines[per_cry][f'{opt2}']
            p_mix = fig.line(ele1_mix, ele2_mix, line_color=colors[0], line_width=linesize, line_dash="dotted", line_alpha=opac, legend_label="P-Rich Mixing Lines")
            p_mix_hover = add_mixing_hover(p_mix)
            fig.add_tools(p_mix_hover)
    if mGroup != 'None':
        try:
            names1, meteorites_opt1, refs_grp, names_un1, ungrp1, refs_ungrp = get_meteorite_coords(opt1, unitx, mGroup)
            names2, meteorites_opt2, _, names_un2, ungrp2, _ = get_meteorite_coords(opt2, unity, mGroup)
            grp_plot = fig.scatter(meteorites_opt1, meteorites_opt2, color="black", alpha=opac, marker='^', size=markerSize, legend_label=mGroup)
            grp_plot.data_source.data['name'] = names1
            grp_plot.data_source.data['ref'] = refs_grp
            grp_hover = HoverTool(renderers=[grp_plot], tooltips=[("(x,y)", "($x, $y)"), ("Meteorite Name", "@name"), ("Reference", "@ref")])            
            fig.add_tools(grp_hover)
            
            if type(ungrp1) != NoneType and type(ungrp2) != NoneType:
                if np.nansum(ungrp1)>0 and np.nansum(ungrp2)>0:
                    if mGroup == "IIAB":    
                        ungrp_plot = fig.scatter(ungrp1, ungrp2, color=colors[9], marker='o', size=markerSize, legend_label='IIG')
                        ungrp_plot.data_source.data['name'] = names_un1
                        ungrp_plot.data_source.data['ref'] = refs_ungrp
                        ungrp_hover = HoverTool(renderers=[ungrp_plot], tooltips=[("(x,y)", "($x, $y)"), ("Meteorite Name", "@name"), ("Reference", "@ref")])              
                        fig.add_tools(ungrp_hover)   
                    elif mGroup=="IIIAB":                        
                        sHL_name = np.where(names_un1 == "sHL")[0][0]
                        sHH_name = np.where(names_un1 == "sHH")[0][0]
                        
                        ungrp_plot=fig.scatter(ungrp1[sHL_name:sHH_name], ungrp2[sHL_name:sHH_name], color=colors[9], alpha=opac, marker='o',
                                    size=markerSize, legend_label='IAB sHL')
                        ungrp_plot1=fig.scatter(ungrp1[sHH_name:], ungrp2[sHH_name:], color=colors[3], alpha=opac, marker='diamond',
                                    size=markerSize*1.2, legend_label='IAB sHH')

                        ungrp_plot.data_source.data['name'] = names_un1[sHL_name:sHH_name]
                        ungrp_plot.data_source.data['ref'] = refs_ungrp[sHL_name:sHH_name]
                        ungrp_hover = HoverTool(renderers=[ungrp_plot], tooltips=[("(x,y)", "($x, $y)"), ("Meteorite Name", "@name"), ("Reference", "@ref")])              
                        fig.add_tools(ungrp_hover)
                        
                        ungrp_plot1.data_source.data['name'] = names_un1[sHH_name:]
                        ungrp_plot1.data_source.data['ref'] = refs_ungrp[sHH_name:]
                        ungrp_hover1 = HoverTool(renderers=[ungrp_plot1], tooltips=[("(x,y)", "($x, $y)"), ("Meteorite Name", "@name"), ("Reference", "@ref")])              
                        fig.add_tools(ungrp_hover1)

                    else:
                        ungrp_plot = fig.scatter(ungrp1, ungrp2, color=colors[9], marker='o', size=markerSize, legend_label='Ungrouped')
    
                        ungrp_plot.data_source.data['name'] = names_un1
                        ungrp_plot.data_source.data['ref'] = refs_ungrp
                        ungrp_hover = HoverTool(renderers=[ungrp_plot], tooltips=[("(x,y)", "($x, $y)"), ("Meteorite Name", "@name"), ("Reference", "@ref")])              
                        fig.add_tools(ungrp_hover)        
        except:
            print(f'{opt1} or {opt2} not available for meteorite group, {mGroup}')

    # custom_points_plot = fig.scatter(x='x',  y='y', source=custom_source,color="firebrick", marker="star",size=markerSize)
    # fig.add_tools(HoverTool(renderers=[custom_points_plot],tooltips=[("Meteorite", "@name"), ("x", "@x"), ("y", "@y"),]))

    fig.legend.ncols=numCols
    fig.add_layout(fig.legend[0], 'above')
    fig.legend.click_policy="hide"
    fig.legend.location="center"
    
    return fig


def convert_to_wt(comp, units):
    if units == 'µg/g' or units == 'ppm':
        comp = comp/1e4
    elif units == 'mg/g':
        comp = comp/10
    elif units == 'ng/g':
        comp = comp/1e7

    return comp

In [5]:
from bokeh.io import reset_output
import os
from model.functions import get_model_path, get_default_comp, read_sheet

# Run crystallization to update plot information
def update_two_crystallization(ele1, ele2, lm01, lm02, unit1, unit2, s_lm0, p_lm0, ni_lm0, f_init, save_data, liq_imm_case):
    model_path = get_model_path(s_lm0, p_lm0, ni_lm0, f_init, liq_imm_case)
    try:
        state_data = pd.read_excel(model_path, sheet_name='State Data')
        data1 = read_sheet(model_path, ele1, lm01, unit1)
    except:
        state_data, data1 = run_crystallization(ele1, lm01, unit1, s_lm0, p_lm0, ni_lm0, f_init, save_data, liq_imm_case)

    try:
        data2 = read_sheet(model_path, ele2, lm02, unit2)
    except:
        _, data2 = run_crystallization(ele2, lm02, unit2, s_lm0, p_lm0, ni_lm0, f_init, save_data, liq_imm_case)
            
    return state_data, data1, data2 

def update_three_crystallization(xele, ele1, ele2, xlm0, lm01, lm02, unitx, unit1, unit2, s_lm0, p_lm0, ni_lm0, f_init, save_data, liq_imm_case):
    model_path = get_model_path(s_lm0, p_lm0, ni_lm0, f_init, liq_imm_case)
    try:
        state_data = pd.read_excel(model_path, sheet_name='State Data')
        data1 = read_sheet(model_path, ele1, lm01, unit1)
    except:
        state_data, data1 = run_crystallization(ele1, lm01, unit1, s_lm0, p_lm0, ni_lm0, f_init, save_data, liq_imm_case)

    try:
        data2 = read_sheet(model_path, ele2, lm02, unit2)
    except:
        _, data2 = run_crystallization(ele2, lm02, unit2, s_lm0, p_lm0, ni_lm0, f_init, save_data, liq_imm_case)
    
    try:
        datax = read_sheet(model_path, xele, xlm0, unitx)
    except:
        _, datax = run_crystallization(xele, xlm0, unitx, s_lm0, p_lm0, ni_lm0, f_init, save_data, liq_imm_case)
            
    return state_data, datax, data1, data2 

In [6]:
#### from bokeh.io import reset_output
import os
from model.functions import get_model_path, get_default_comp
from ipysheet import sheet, cell, row, column, cell_range, to_dataframe, from_dataframe


# Style for widgets
style = {'description_width': 'initial'}

element_options = ['Ag', 'As', 'Au', 'Bi', 'Co', 'Cr', 'Cu', 'Ga', 'Ge', 'Ir', 'Mo', 'Ni', 'Os', 'P', 'Pb', 'Pd', 'Pt', 'Re', 'Rh', 'Ru', 'Sb', 'Sn', 'V', 'W', 'Zn']
unit_options = ['ppm', 'wt%', 'mg/g', 'ppb']
# Create the dropdowns and float widgets
xele = widgets.Dropdown(options=element_options, value='As', description='X Element', disabled=False, style=style)
yele = widgets.Dropdown(options=element_options, value='Ir', description='Y Element:',disabled=False, style=style)
y2ele = widgets.Dropdown(options=element_options, value='Ir', description='Y Element (denominator):', disabled=False, style=style)
group = widgets.Dropdown(options=['None', 'IC', 'IIAB', 'IIIAB', 'IIIE', 'IVA', 'IIC', 'IID', 'IIF', 'IIIF', 'IVB', 'SBT'],
    value='IIIAB',description='Meteorite Group:', disabled=False, style=style)

unit1 = widgets.Dropdown(options=unit_options,value='ppm', description='Unit:', disabled=False, style=style)
unit2 = widgets.Dropdown(options= unit_options, value='ppm',description='Unit:', disabled=False, style=style)
unit3 = widgets.Dropdown(options=unit_options, value='ppm',description='Unit:', disabled=False, style=style)

mixing_lines_dropdown = widgets.Dropdown(options=['Default', 'Fixed Number', 'At Percent(s) Crystallization'],
    value='Default', description='Mixing Line Options:', disabled=False,style=style)


x_lm0 = widgets.FloatText(value=7.80, step=0.1, description='Initial:', readout_format='.2f', style=style)
y_lm0 = widgets.FloatText(value=3.70, step=0.1, description='Initial:', readout_format='.2f', style=style)
y2_lm0 = widgets.FloatText(value=3.70, step=0.1, description='Initial:', readout_format='.2f', style=style)
LMS = widgets.FloatText(value=9, step=0.1, description='S Initial (wt%):', readout_format='.2f', style=style)
LMP = widgets.FloatText(value=0.3, step=0.1, description='P Initial (wt%):', readout_format='.2f', style=style)
LMNi = widgets.FloatText(value=7.27, step=0.1, description='Ni Initial (wt%):', readout_format='.2f', style=style)
f0 = widgets.FloatText(value=0.0005, description='Initial Step Size:', disabled=False, readout_format='.5f', style = style)
num_lines_box = widgets.BoundedIntText(value=0, min=0, max=200,step=1, description='Number of Mixing Lines:', disabled=False, style=style)
custom_entries = widgets.BoundedIntText(value=0, min=0, max=50, description='Custom Meteorite Entries:',disabled=False, style=style)

per_cry_text = widgets.Textarea(placeholder='1, 10, 30', description='Percents (comma seperated):', disabled=False, style=style)
num_lines_box.layout.visibility = 'hidden' 
per_cry_text.layout.visibility = 'hidden' 

cases = widgets.Dropdown(options=['No Liquid Immiscibility', 'Equilibrium', 'S-Rich Crystalliation', 'P-Rich Crystallization'],
    value='Equilibrium', description='Case', disabled=False, style=style)

save_check = widgets.Checkbox(value=True,description='Save Data', disabled=False, indent=False)
# plot_type_toggle = widgets.ToggleButtons(options=['Element-Element', 'Ratio'], description='Plot Type:', disabled=False, button_style='', 
#     tooltips=['Elemental trend plot', 'Ratio of two elements vs element'])
plot_type_toggle = widgets.ToggleButtons(options=['Element-Element'], description='Plot Type:', disabled=True, button_style='', 
    tooltips=['Elemental trend plot'])

# Arrange widgets
row0 = widgets.HBox([plot_type_toggle])
row1 = widgets.HBox([xele, x_lm0, unit1])
row2 = widgets.HBox([yele, y_lm0, unit2])
row2a = widgets.HBox([y2ele, y2_lm0, unit3])
row2a.layout.visibility = 'hidden'
row3 = widgets.HBox([LMS, LMP, LMNi])
row4 = widgets.HBox([f0, group, cases, save_check])

# Create a button to trigger the update
update_button = widgets.Button(description="Update Plot")
default_button = widgets.Button(description="Default Values")
entries_button = widgets.Button(description="Add Custom Points")

state_data, data1, data2 = update_two_crystallization(xele.value, yele.value, x_lm0.value, y_lm0.value, unit1.value, unit2.value, LMS.value, LMP.value, LMNi.value, f0.value, save_check.value, liq_imm_case=cases_dict[cases.value])

row5 = widgets.HBox([update_button, default_button, mixing_lines_dropdown, num_lines_box, per_cry_text])


def update_defaults(button):
    lookup_file = './data/start_comp_lookup_default.csv'
    ele1_start, ele1_unit = get_default_comp(xele.value, group.value, lookup_file)
    ele2_start, ele2_unit = get_default_comp(yele.value, group.value, lookup_file)
    ele3_start, ele3_unit = get_default_comp(y2ele.value, group.value, lookup_file)

    x_lm0.value = ele1_start
    unit1.value = ele1_unit
    y_lm0.value = ele2_start
    unit2.value = ele2_unit
    y2_lm0.value = ele3_start
    unit3.value = ele3_unit

    s_start, s_unit = get_default_comp('S', group.value, lookup_file)
    p_start, p_unit = get_default_comp('P', group.value, lookup_file)
    ni_start, ni_unit = get_default_comp('Ni', group.value, lookup_file)

    s_start = convert_to_wt(s_start, s_unit)
    p_start = convert_to_wt(p_start, p_unit)
    ni_start = convert_to_wt(ni_start, ni_unit)

    LMS.value = round(s_start, 2)
    LMP.value = round(p_start, 2)
    LMNi.value = round(ni_start, 2)

    
def mixing_line_change(change):
    with out:
        if change["new"] == 'Fixed Number':
            num_lines_box.layout.visibility = 'visible' 
            per_cry_text.layout.visibility = 'hidden' 
        elif change["new"] == 'Default':
            num_lines_box.layout.visibility = 'hidden' 
            per_cry_text.layout.visibility = 'hidden' 
        else:
            num_lines_box.layout.visibility = 'hidden' 
            per_cry_text.layout.visibility = 'visible' 

def change_plot_type(change):
    with out:
        if change["new"] == "Element-Element":
            row2a.layout.visibility='hidden'   
            yele.description = 'Y Element:'                    
        if change["new"] == "Ratio":
            row2a.layout.visibility='visible'   
            yele.description = 'Y Element (numerator):'    

custom_sheet = sheet(rows=custom_entries.value+1, columns=3, column_headers=False)
cell(0, 0, "Name", read_only=True)
cell(0, 1, f'{xele.value} ({unit1.value})', read_only=True)
cell(0, 2, f'{yele.value} ({unit2.value})', read_only=True)
old_rows = 1

def change_num_entries(change):
    global old_rows, custom_sheet
    with out:        
        new_rows = custom_entries.value + 1  
        if new_rows > old_rows:
            for r in range(old_rows, new_rows):
                cell(r, 0, "")
                cell(r, 1, "")
                cell(r, 2, "")
        else:
            out.clear_output()        
            # custom_points_plot.legend_label="Custom Points"
            df = to_dataframe(custom_sheet)
                 
            custom_x = pd.to_numeric(df.iloc[1:new_rows, 1], errors="coerce").tolist()
            custom_y = pd.to_numeric(df.iloc[1:new_rows, 2], errors="coerce").tolist()
            
            names = df.iloc[1:new_rows,0].astype(str).tolist()
            custom_points_plot.data_source.data['name'] = names
            custom_points_plot.data_source.data['x'] = custom_x
            custom_points_plot.data_source.data['y'] = custom_y
        
            # custom_points_plot = fig.scatter(custom_x, custom_y, color="firebrick", marker="star",size=10, legend_label="Custom Points")
            # custom_points_plot.data_source.data['name'] = names
            # fig.add_tools(HoverTool(renderers=[custom_points_plot],tooltips=[("Meteorite", "@name"), ("x", "@x"), ("y", "@y"),]))
            show(fig)
        
        custom_sheet.rows = new_rows 
        old_rows = new_rows


def add_entries(button):
    global custom_points_plot
    with out:
        out.clear_output()        
        # custom_points_plot.legend_label="Custom Points"
        df = to_dataframe(custom_sheet)
             
        custom_x = pd.to_numeric(df.iloc[1:, 1], errors="coerce").tolist()
        custom_y = pd.to_numeric(df.iloc[1:, 2], errors="coerce").tolist()
        
        names = df.iloc[1:,0].astype(str).tolist()

        custom_points_plot = fig.scatter(custom_x, custom_y, color="firebrick", marker="star",size=10, legend_label="Custom Points")
        custom_points_plot.data_source.data['name'] = names
        fig.add_tools(HoverTool(renderers=[custom_points_plot],tooltips=[("Meteorite", "@name"), ("x", "@x"), ("y", "@y"),]))
        show(fig)

def change_sheet_labels(change):
        cell(0, 1, f'{xele.value} ({unit1.value})', read_only=True)
        cell(0, 2, f'{yele.value} ({unit2.value})', read_only=True)


def update_crystallization(button):
    global fig
    # out.clear_output()
    # reset_output()
    with out:
        out.clear_output()
        if plot_type_toggle.value == 'Element-Element':
            state_data, data1, data2 = update_two_crystallization(xele.value, yele.value, x_lm0.value, y_lm0.value, unit1.value, unit2.value, LMS.value, LMP.value, LMNi.value,
                                                      f0.value, save_check.value, liq_imm_case=cases_dict[cases.value])
            # display(ui)
            fig = generate_bokeh_plot(state_data, data1, data2, xele, yele, unit1, unit2, group, mixing_lines_dropdown, liq_imm_case=cases_dict[cases.value])
            # show(fig)
        elif plot_type_toggle.value == 'Ratio':
            state_data, xdata, data1, data2 = update_three_crystallization(xele.value, yele.value, y2ele.value, x_lm0.value, y_lm0.value, y2_lm0.value, 
                                                                           unitx.value, unit1.value, unit2.value, LMS.value,
                                                             LMP.value, LMNi.value, f0.value, save_check.value, liq_imm_case=cases_dict[cases.value])
            # display(ui)
            fig = generate_ratio_plot(state_data, xdata, data1, data2, xele, yele, y2ele, unit1, unit2, unit3, group,mixing_lines_dropdown) 
            # show(fig)
            
        entries_button._click_handlers(entries_button)

xele.observe(change_sheet_labels, names='value')
yele.observe(change_sheet_labels, names='value')
unit1.observe(change_sheet_labels, names='value')
unit2.observe(change_sheet_labels, names='value')

update_button.on_click(update_crystallization)
default_button.on_click(update_defaults)
mixing_lines_dropdown.observe(mixing_line_change, names='value')
plot_type_toggle.observe(change_plot_type, names='value')
custom_entries.observe(change_num_entries, names='value')
entries_button.on_click(add_entries)

# Place the rows vertically
out = widgets.Output()
plot_adjacent_col = widgets.VBox([custom_entries, custom_sheet, entries_button])
row6 = widgets.HBox([out, plot_adjacent_col])
ui = widgets.VBox([row0, row1, row2, row2a, row3, row4, row5, row6])
display(ui)
# display(out)

# update_defaults(None)
with out:
    fig = generate_bokeh_plot(state_data, data1, data2, xele, yele, unit1, unit2, group, mixing_lines_dropdown, liq_imm_case=cases_dict[cases.value])
    show(fig)
